#========================================
# CELL 1 - Check the GPU
#========================================

In [ ]:
!nvidia-smi

Mon Aug 10 06:58:37 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   53C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

#========================================
# CELL 2 - Connect Google Drive
# ========================================

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


#========================================
# CELL 3 - Create our Qwen3-TTS workspace
#========================================


In [ ]:
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/Qwen3TTS")

MODELS_DIR = DRIVE_ROOT / "models"
VOICES_DIR = DRIVE_ROOT / "voices"
OUTPUT_DIR = DRIVE_ROOT / "output"

MODELS_DIR.mkdir(parents=True, exist_ok=True)
VOICES_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Qwen3-TTS workspace:")
print(DRIVE_ROOT)

Qwen3-TTS workspace:
/content/drive/MyDrive/Qwen3TTS


#======================================
# CELL 4 -  Install Qwen3-**TTS**
#======================================

In [ ]:
!pip install -U qwen-tts

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.4/61.4 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 kB 6.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of gradio to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.5/113.5 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.9/380.9 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 105.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.3/32.3 MB 69.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 94.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 46.1 MB/s eta 0:00:00
  Created wheel for sox: filename=sox-1.5.0-py3-none-any.whl size=40036 sha256=ce5bb423983543df

#======================================
# CELL 5 - Check that Qwen imports
#======================================

In [ ]:
import torch
import soundfile as sf
from qwen_tts import Qwen3TTSModel

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


    If you do not have SoX, proceed here:
     - - - http://sox.sourceforge.net/ - - -

    If you do (or think that you should) have SoX, double-check your
    path variables.
    



********
********
 
PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


#======================================
# CELL 6 - Load the VoiceDesign model
#======================================


In [ ]:
import torch
from qwen_tts import Qwen3TTSModel

design_model = Qwen3TTSModel.from_pretrained(
    "Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign",
    device_map="cuda:0",
    dtype=torch.bfloat16,
)

print("VoiceDesign model loaded.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.83G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/245 [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

...

preprocessor_config.json:   0%|          | 0.00/127 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

VoiceDesign model loaded.


# ======================================
# CELL 7 — DEFINE & DESIGN YOUR VOICE
#======================================

# Give your voice a permanent name.
# This will become the folder/name we use in Google Drive later.

In [ ]:
VOICE_NAME = "cold_analytical_male"

# ------------------------------------------------------------
# 1. Describe the voice you want
# ------------------------------------------------------------

voice_description = """
Young adult male voice.
Low to medium-low pitch.
Calm, controlled and emotionally restrained.
Intelligent and analytical.
Measured speaking pace.
Very clear and precise pronunciation.
Quiet confidence.
Slightly detached and observant.
Subtle seriousness rather than theatrical darkness.
Natural breathing.
Minimal unnecessary emotional emphasis.
The voice should sound original and distinctive.
"""


# ------------------------------------------------------------
# 2. Give Qwen some text to speak while creating the voice
# ------------------------------------------------------------

reference_text = """
Most people think they understand what they see.
Usually, they only understand what they expect to see.
There is a difference.
"""


# ------------------------------------------------------------
# 3. Choose the language
# ------------------------------------------------------------

VOICE_LANGUAGE = "English"


# ------------------------------------------------------------
# Show our settings before we generate anything
# ------------------------------------------------------------

print("VOICE NAME:")
print(VOICE_NAME)

print("\nVOICE DESCRIPTION:")
print(voice_description)

print("\nREFERENCE TEXT:")
print(reference_text)

print("\nLANGUAGE:")
print(VOICE_LANGUAGE)

print("\n" + "=" * 60)
print("Ready to generate.")
print("=" * 60)

VOICE NAME:
cold_analytical_male

VOICE DESCRIPTION:

Young adult male voice.
Low to medium-low pitch.
Calm, controlled and emotionally restrained.
Intelligent and analytical.
Measured speaking pace.
Very clear and precise pronunciation.
Quiet confidence.
Slightly detached and observant.
Subtle seriousness rather than theatrical darkness.
Natural breathing.
Minimal unnecessary emotional emphasis.
The voice should sound original and distinctive.


REFERENCE TEXT:

Most people think they understand what they see.
Usually, they only understand what they expect to see.
There is a difference.


LANGUAGE:
English

Ready to generate.


# =========================================
# CELL 8 — Generate, Listen & Approve Voice
# =========================================

In [ ]:
# ============================================================
# CELL 8 — VOICE DESIGN STUDIO
# ============================================================

import soundfile as sf
from IPython.display import Audio, display
from pathlib import Path

print("\n" + "=" * 60)
print("QWEN3-TTS — VOICE DESIGN STUDIO")
print("=" * 60)

print(f"\nDesigning voice: {VOICE_NAME}")
print("This may take a little while on the T4.\n")

# ------------------------------------------------------------
# Generate the voice
# ------------------------------------------------------------

wavs, sr = design_model.generate_voice_design(
    text=reference_text,
    language=VOICE_LANGUAGE,
    instruct=voice_description,
)

# ------------------------------------------------------------
# Save temporary preview
# ------------------------------------------------------------
preview_path = Path("/content/voice_design_preview.wav")

sf.write(
    preview_path,
    wavs[0],
    sr
)

print("Voice generated successfully.")
print("\nListen to the result below:\n")

display(Audio(str(preview_path)))

# ------------------------------------------------------------
# Ask whether the voice is acceptable
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("VOICE REVIEW")
print("=" * 60)

while True:

    answer = input(
        "\nDo you like this voice? "
        "[Y = keep it / N = redesign]: "
    ).strip().lower()

    if answer in ["y", "yes"]:
        approved = True
        break

    elif answer in ["n", "no"]:
        approved = False
        break

    else:
        print("Please enter Y or N.")


# ------------------------------------------------------------
# Handle result
# ------------------------------------------------------------

if approved:

    print("\n" + "=" * 60)
    print("VOICE APPROVED")
    print("=" * 60)

    print(f"""
Excellent.

"{VOICE_NAME}" has been approved.

The next step will be to:
1. Save the reference audio to Google Drive.
2. Save the voice description.
3. Save the reference text.
4. Load Qwen3-TTS Base.
5. Create the reusable voice-clone prompt.
6. Save that reusable voice for future TTS.

We will do those steps in the next cell.
""")

else:

    print("\n" + "=" * 60)
    print("VOICE REJECTED")
    print("=" * 60)

    print("""
      No problem.

      Go back to CELL 7 and modify:

          voice_description

      You can change things such as:

          - pitch
          - age
          - speaking speed
          - emotional intensity
          - warmth
          - breathiness
          - confidence
          - seriousness
          - pronunciation
          - energy
          - vocal character

      Then run CELL 7 again, followed by CELL 8.

      The current voice will NOT be saved as a permanent voice.
      """)

Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.



QWEN3-TTS — VOICE DESIGN STUDIO

Designing voice: cold_analytical_male
This may take a little while on the T4.

Voice generated successfully.

Listen to the result below:




VOICE REVIEW

Do you like this voice? [Y = keep it / N = redesign]: y

VOICE APPROVED

Excellent.

"cold_analytical_male" has been approved.

The next step will be to:
1. Save the reference audio to Google Drive.
2. Save the voice description.
3. Save the reference text.
4. Load Qwen3-TTS Base.
5. Create the reusable voice-clone prompt.
6. Save that reusable voice for future TTS.

We will do those steps in the next cell.



# ============================================================
# CELL 9 — SAVE APPROVED VOICE + FREE VOICEDESIGN MODEL
# ============================================================


In [ ]:
from pathlib import Path
import shutil
import gc
import torch

# ------------------------------------------------------------
# 1. Create permanent voice folder
# ------------------------------------------------------------

VOICE_DIR = VOICES_DIR / VOICE_NAME
VOICE_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 60)
print("SAVING APPROVED VOICE")
print("=" * 60)

print(f"\nVoice folder:")
print(VOICE_DIR)

# ------------------------------------------------------------
# 2. Save generated reference audio
# ------------------------------------------------------------

reference_path = VOICE_DIR / "reference.wav"

shutil.copy2(
    "/content/voice_design_preview.wav",
    reference_path
)

print(f"\nSaved audio:")
print(reference_path)

# ------------------------------------------------------------
# 3. Save voice description
# ------------------------------------------------------------

description_path = VOICE_DIR / "description.txt"

description_path.write_text(
    voice_description.strip(),
    encoding="utf-8"
)

print(f"Saved description:")
print(description_path)

# ------------------------------------------------------------
# 4. Save reference text
# ------------------------------------------------------------

reference_text_path = VOICE_DIR / "reference_text.txt"

reference_text_path.write_text(
    reference_text.strip(),
    encoding="utf-8"
)

print(f"Saved reference text:")
print(reference_text_path)

# ------------------------------------------------------------
# 5. Verify files
# ------------------------------------------------------------

print("\nSaved files:")

for path in VOICE_DIR.iterdir():
    print(f"  ✓ {path.name}")

# ------------------------------------------------------------
# 6. Free VoiceDesign model from GPU
# ------------------------------------------------------------

print("\nFreeing VoiceDesign model from GPU...")

del design_model
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

print("✓ VoiceDesign model released.")
print("✓ CUDA cache cleared.")

print("\n" + "=" * 60)
print("CELL 9 COMPLETE")
print("=" * 60)

SAVING APPROVED VOICE

Voice folder:
/content/drive/MyDrive/Qwen3TTS/voices/cold_analytical_male

Saved audio:
/content/drive/MyDrive/Qwen3TTS/voices/cold_analytical_male/reference.wav
Saved description:
/content/drive/MyDrive/Qwen3TTS/voices/cold_analytical_male/description.txt
Saved reference text:
/content/drive/MyDrive/Qwen3TTS/voices/cold_analytical_male/reference_text.txt

Saved files:
  ✓ reference.wav
  ✓ description.txt
  ✓ reference_text.txt

Freeing VoiceDesign model from GPU...
✓ VoiceDesign model released.
✓ CUDA cache cleared.

CELL 9 COMPLETE


# ============================================================
# CELL 10 — LOAD QWEN3-TTS BASE MODEL
# ============================================================


In [ ]:
import torch
from qwen_tts import Qwen3TTSModel

print("=" * 60)
print("LOADING QWEN3-TTS BASE MODEL")
print("=" * 60)

clone_model = Qwen3TTSModel.from_pretrained(
    "Qwen/Qwen3-TTS-12Hz-1.7B-Base",
    device_map="cuda:0",
    dtype=torch.bfloat16,
)

print("\n✓ Base model loaded.")
print("✓ Ready to create reusable voice clone prompt.")

LOADING QWEN3-TTS BASE MODEL


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/245 [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

...

preprocessor_config.json:   0%|          | 0.00/127 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]


✓ Base model loaded.
✓ Ready to create reusable voice clone prompt.


# ============================================================
# CELL 11 — CREATE REUSABLE VOICE CLONE PROMPT
# ============================================================


In [ ]:
print("=" * 60)
print("CREATING REUSABLE VOICE CLONE PROMPT")
print("=" * 60)

reference_path = VOICE_DIR / "reference.wav"
reference_text_path = VOICE_DIR / "reference_text.txt"

reference_text_for_clone = reference_text_path.read_text(
    encoding="utf-8"
).strip()

print("\nReference audio:")
print(reference_path)

print("\nReference text:")
print(reference_text_for_clone)

print("\nCreating voice clone prompt...")

voice_clone_prompt = clone_model.create_voice_clone_prompt(
    ref_audio=str(reference_path),
    ref_text=reference_text_for_clone,
)

print("\n✓ Voice clone prompt created.")
print("Type:", type(voice_clone_prompt))
print("Items:", len(voice_clone_prompt))

CREATING REUSABLE VOICE CLONE PROMPT

Reference audio:
/content/drive/MyDrive/Qwen3TTS/voices/cold_analytical_male/reference.wav

Reference text:
Most people think they understand what they see.
Usually, they only understand what they expect to see.
There is a difference.

Creating voice clone prompt...

✓ Voice clone prompt created.
Type: <class 'list'>
Items: 1


# ============================================================
# CELL 12 — INSPECT VOICE CLONE PROMPT STRUCTURE
# ============================================================


In [ ]:
prompt_item = voice_clone_prompt[0]

print("=" * 60)
print("VOICE CLONE PROMPT INSPECTION")
print("=" * 60)

print("\nTop-level type:")
print(type(voice_clone_prompt))

print("\nPrompt item type:")
print(type(prompt_item))

print("\nAttributes:")

for name in dir(prompt_item):
    if not name.startswith("_"):
        try:
            value = getattr(prompt_item, name)
            print(f"\n{name}:")
            print("  type =", type(value))

            if hasattr(value, "shape"):
                print("  shape =", value.shape)

            elif isinstance(value, (str, int, float, bool, type(None))):
                print("  value =", value)

            elif isinstance(value, (list, tuple, dict)):
                print("  length =", len(value))

        except Exception as e:
            print("  error reading attribute:", e)

VOICE CLONE PROMPT INSPECTION

Top-level type:
<class 'list'>

Prompt item type:
<class 'qwen_tts.inference.qwen3_tts_model.VoiceClonePromptItem'>

Attributes:

icl_mode:
  type = <class 'bool'>
  value = True

ref_code:
  type = <class 'torch.Tensor'>
  shape = torch.Size([108, 16])

ref_spk_embedding:
  type = <class 'torch.Tensor'>
  shape = torch.Size([2048])

ref_text:
  type = <class 'str'>
  value = Most people think they understand what they see.
Usually, they only understand what they expect to see.
There is a difference.

x_vector_only_mode:
  type = <class 'bool'>
  value = False


# ============================================================
# CELL 13 — SAVE REUSABLE VOICE CLONE PROMPT
# ============================================================


In [ ]:
import torch

PROMPT_PATH = VOICE_DIR / "voice_clone_prompt.pt"

prompt_item = voice_clone_prompt[0]

# Move tensors to CPU before saving.
# This makes the saved file independent of the current GPU.
saved_prompt = {
    "icl_mode": prompt_item.icl_mode,
    "ref_code": prompt_item.ref_code.detach().cpu(),
    "ref_spk_embedding": prompt_item.ref_spk_embedding.detach().cpu(),
    "ref_text": prompt_item.ref_text,
    "x_vector_only_mode": prompt_item.x_vector_only_mode,
}

torch.save(saved_prompt, PROMPT_PATH)

print("=" * 60)
print("VOICE CLONE PROMPT SAVED")
print("=" * 60)

print(f"\nSaved to:")
print(PROMPT_PATH)

print("\nSaved contents:")

for key, value in saved_prompt.items():
    if isinstance(value, torch.Tensor):
        print(f"  ✓ {key}: Tensor {tuple(value.shape)}")
    else:
        print(f"  ✓ {key}: {value}")

print("\n✓ Reusable voice clone prompt saved.")

VOICE CLONE PROMPT SAVED

Saved to:
/content/drive/MyDrive/Qwen3TTS/voices/cold_analytical_male/voice_clone_prompt.pt

Saved contents:
  ✓ icl_mode: True
  ✓ ref_code: Tensor (108, 16)
  ✓ ref_spk_embedding: Tensor (2048,)
  ✓ ref_text: Most people think they understand what they see.
Usually, they only understand what they expect to see.
There is a difference.
  ✓ x_vector_only_mode: False

✓ Reusable voice clone prompt saved.


# ============================================================
# CELL 14 — LOAD & VERIFY SAVED VOICE CLONE PROMPT
# ============================================================


In [ ]:
import torch

print("=" * 60)
print("LOADING SAVED VOICE CLONE PROMPT")
print("=" * 60)

loaded_prompt = torch.load(
    PROMPT_PATH,
    map_location="cpu",
    weights_only=True
)

print("\nLoaded successfully.")

print("\nContents:")

for key, value in loaded_prompt.items():
    if isinstance(value, torch.Tensor):
        print(f"  ✓ {key}: Tensor {tuple(value.shape)}")
    else:
        print(f"  ✓ {key}: {value}")

# ------------------------------------------------------------
# Verify expected structure
# ------------------------------------------------------------

assert loaded_prompt["icl_mode"] is True
# The frame count T varies with the reference-audio duration
# (T = ceil(valid 24 kHz samples / 1920)); only the (T, 16) structure is fixed.
assert loaded_prompt["ref_code"].ndim == 2
assert loaded_prompt["ref_code"].shape[1] == 16
assert loaded_prompt["ref_code"].shape[0] >= 1
assert loaded_prompt["ref_spk_embedding"].shape == (2048,)
assert isinstance(loaded_prompt["ref_text"], str)
assert loaded_prompt["x_vector_only_mode"] is False

print("\n✓ All prompt fields verified.")
print("✓ Saved voice can be restored successfully.")

LOADING SAVED VOICE CLONE PROMPT

Loaded successfully.

Contents:
  ✓ icl_mode: True
  ✓ ref_code: Tensor (108, 16)
  ✓ ref_spk_embedding: Tensor (2048,)
  ✓ ref_text: Most people think they understand what they see.
Usually, they only understand what they expect to see.
There is a difference.
  ✓ x_vector_only_mode: False

✓ All prompt fields verified.
✓ Saved voice can be restored successfully.


# ============================================================
# CELL 15 — RESTORE VOICE CLONE PROMPT OBJECT
# ============================================================


In [ ]:
from qwen_tts.inference.qwen3_tts_model import VoiceClonePromptItem

print("=" * 60)
print("RESTORING VOICE CLONE PROMPT OBJECT")
print("=" * 60)

device = next(clone_model.model.parameters()).device

gpu_prompt = [
    VoiceClonePromptItem(
        ref_code=loaded_prompt["ref_code"].to(device),
        ref_spk_embedding=loaded_prompt["ref_spk_embedding"].to(device),
        x_vector_only_mode=loaded_prompt["x_vector_only_mode"],
        icl_mode=loaded_prompt["icl_mode"],
        ref_text=loaded_prompt["ref_text"],
    )
]

print("\nModel device:", device)

print("\nPrompt object:")
print("  ✓ Type:", type(gpu_prompt[0]))
print("  ✓ Number of prompts:", len(gpu_prompt))
print("  ✓ ref_code:", tuple(gpu_prompt[0].ref_code.shape))
print("  ✓ ref_spk_embedding:", tuple(gpu_prompt[0].ref_spk_embedding.shape))
print("  ✓ ref_text:", gpu_prompt[0].ref_text)
print("  ✓ icl_mode:", gpu_prompt[0].icl_mode)
print("  ✓ x_vector_only_mode:", gpu_prompt[0].x_vector_only_mode)

print("\n✓ VoiceClonePromptItem restored successfully.")

RESTORING VOICE CLONE PROMPT OBJECT

Model device: cuda:0

Prompt object:
  ✓ Type: <class 'qwen_tts.inference.qwen3_tts_model.VoiceClonePromptItem'>
  ✓ Number of prompts: 1
  ✓ ref_code: (108, 16)
  ✓ ref_spk_embedding: (2048,)
  ✓ ref_text: Most people think they understand what they see.
Usually, they only understand what they expect to see.
There is a difference.
  ✓ icl_mode: True
  ✓ x_vector_only_mode: False

✓ VoiceClonePromptItem restored successfully.


# ============================================================
# CELL 16 — FIRST VOICE CLONE TEST
# ============================================================


In [ ]:
# ============================================================
# CELL 16 — FIRST VOICE CLONE TEST
# ============================================================


In [ ]:
import soundfile as sf
import torch

TEST_TEXT = (
    "This is a test of the cold analytical voice. "
    "The delivery should sound calm, controlled, and deliberate."
)

OUTPUT_PATH = VOICE_DIR / "clone_test_01.wav"

print("=" * 60)
print("GENERATING FIRST VOICE CLONE TEST")
print("=" * 60)

print("\nText:")
print(TEST_TEXT)

print("\nGenerating...")

with torch.inference_mode():
    wavs, sr = clone_model.generate_voice_clone(
        text=TEST_TEXT,
        language="English",
        voice_clone_prompt=gpu_prompt,
    )

sf.write(
    OUTPUT_PATH,
    wavs[0],
    sr
)

print("\n✓ Generation complete.")
print("Saved to:")
print(OUTPUT_PATH)

print("\nSample rate:", sr)
print("Audio samples:", len(wavs[0]))

Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


GENERATING FIRST VOICE CLONE TEST

Text:
This is a test of the cold analytical voice. The delivery should sound calm, controlled, and deliberate.

Generating...

✓ Generation complete.
Saved to:
/content/drive/MyDrive/Qwen3TTS/voices/cold_analytical_male/clone_test_01.wav

Sample rate: 24000
Audio samples: 184320


# ============================================================
# CELL 17 — PLAY FIRST VOICE CLONE TEST
# ============================================================


In [ ]:
from IPython.display import Audio, display

print("=" * 60)
print("PLAYING FIRST VOICE CLONE TEST")
print("=" * 60)

display(Audio(
    str(OUTPUT_PATH),
    autoplay=False
))

PLAYING FIRST VOICE CLONE TEST


# ============================================================
# CELL 18 — LONGER VOICE TEST
# ============================================================


In [ ]:
TEST_TEXT_2 = (
    "Most people think they understand what they see. "
    "But perception is rarely as simple as it seems. "
    "Our minds constantly fill in missing information, "
    "make assumptions, and turn incomplete details into a story. "
    "And sometimes, what we believe we are seeing "
    "has very little to do with what is actually there."
)

OUTPUT_PATH_2 = VOICE_DIR / "clone_test_02.wav"

print("=" * 60)
print("GENERATING LONGER VOICE TEST")
print("=" * 60)

print("\nText:")
print(TEST_TEXT_2)

print("\nGenerating...")

with torch.inference_mode():
    wavs, sr = clone_model.generate_voice_clone(
        text=TEST_TEXT_2,
        language="English",
        voice_clone_prompt=gpu_prompt,
    )

sf.write(
    OUTPUT_PATH_2,
    wavs[0],
    sr
)

print("\n✓ Generation complete.")
print("Saved to:")
print(OUTPUT_PATH_2)

print("\nSample rate:", sr)
print("Audio samples:", len(wavs[0]))

Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


GENERATING LONGER VOICE TEST

Text:
Most people think they understand what they see. But perception is rarely as simple as it seems. Our minds constantly fill in missing information, make assumptions, and turn incomplete details into a story. And sometimes, what we believe we are seeing has very little to do with what is actually there.

Generating...

✓ Generation complete.
Saved to:
/content/drive/MyDrive/Qwen3TTS/voices/cold_analytical_male/clone_test_02.wav

Sample rate: 24000
Audio samples: 403200


# ============================================================
# CELL 19 — PLAY LONGER VOICE TEST
# ============================================================


In [ ]:
from IPython.display import Audio, display

display(Audio(
    str(OUTPUT_PATH_2),
    autoplay=False
))

# ============================================
# CELL 20 — REUSABLE NARRATION GENERATOR
# ============================================

## What this does:
This will be our first proper narration tool. You will eventually paste a script into NARRATION_TEXT, and the cell will use your saved cold analytical voice clone to generate the narration and save it to your Drive.

For now, we'll keep it simple and generate one piece of narration at a time. We won't deal with long-script chunking yet.

## Expected result:
A WAV file should appear inside:
```text
Qwen3TTS/
└── voices/
    └── cold_analytical_male/
        └── narrations/
            └── narration_01.wav
```

In [ ]:
from pathlib import Path
import soundfile as sf
import torch

# ------------------------------------------------------------
# 1. Narration text
# ------------------------------------------------------------

NARRATION_TEXT = """
Most people think they understand what they see.
But perception is rarely as simple as it seems.
Our minds constantly fill in missing information,
make assumptions, and turn incomplete details into a story.
"""

# ------------------------------------------------------------
# 2. Create narration output folder
# ------------------------------------------------------------

NARRATION_DIR = VOICE_DIR / "narrations"
NARRATION_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PATH = NARRATION_DIR / "narration_01.wav"

# ------------------------------------------------------------
# 3. Generate narration
# ------------------------------------------------------------

print("=" * 60)
print("GENERATING NARRATION")
print("=" * 60)

print("\nText:")
print(NARRATION_TEXT.strip())

print("\nGenerating...")

with torch.inference_mode():
    wavs, sr = clone_model.generate_voice_clone(
        text=NARRATION_TEXT.strip(),
        language="English",
        voice_clone_prompt=gpu_prompt,
    )

# ------------------------------------------------------------
# 4. Save audio
# ------------------------------------------------------------

sf.write(
    OUTPUT_PATH,
    wavs[0],
    sr
)

print("\n✓ Narration generated successfully.")

print("\nSaved to:")
print(OUTPUT_PATH)

print("\nSample rate:", sr)
print("Audio samples:", len(wavs[0]))
print("Duration:", round(len(wavs[0]) / sr, 2), "seconds")

Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


GENERATING NARRATION

Text:
Most people think they understand what they see.
But perception is rarely as simple as it seems.
Our minds constantly fill in missing information,
make assumptions, and turn incomplete details into a story.

Generating...

✓ Narration generated successfully.

Saved to:
/content/drive/MyDrive/Qwen3TTS/voices/cold_analytical_male/narrations/narration_01.wav

Sample rate: 24000
Audio samples: 339840
Duration: 14.16 seconds


# ============================================
# CELL 21 — NARRATION PLAYBACK
# ============================================
## What this does:

This cell simply plays the narration generated by Cell 20 directly inside Google Colab.

It does not generate anything new or use the GPU. It only loads the WAV file we already created and gives you an audio player.

## Expected result:

An audio player should appear below the cell. You can press Play and listen to the narration.

In [ ]:
from IPython.display import Audio, display

print("=" * 60)
print("PLAYING GENERATED NARRATION")
print("=" * 60)

display(
    Audio(
        str(OUTPUT_PATH),
        autoplay=False
    )
)

PLAYING GENERATED NARRATION


# ============================================

# CELL 22 — LONG SCRIPT INPUT & VALIDATION

# ============================================

## What this does:

This cell prepares us for long documentary scripts.

Instead of immediately sending a huge script to the TTS model, we'll first put the complete script into FULL_SCRIPT and check its basic information.

This is important because later we'll split long scripts into smaller chunks before generating audio. That will make the process more reliable and prevent one huge generation from consuming too much GPU memory.

For this first step, nothing will be generated and no audio will be created.

## Expected result:

You should see:

- The number of characters in your script
- The approximate number of words
- A preview of the beginning of the script
- A message confirming that the script is ready for chunking

In [ ]:
# ------------------------------------------------------------
# 1. Enter your full narration script here
# ------------------------------------------------------------

FULL_SCRIPT = """
Most people think they understand what they see.

But perception is rarely as simple as it seems.
Our minds constantly fill in missing information,
make assumptions, and turn incomplete details into a story.

This is where things become interesting.
Because sometimes, what we believe we are seeing
has very little to do with what is actually there.

And the difference between perception and reality
can change everything.
"""

# ------------------------------------------------------------
# 2. Basic validation
# ------------------------------------------------------------

script = FULL_SCRIPT.strip()

if not script:
    raise ValueError("FULL_SCRIPT is empty. Please enter your narration.")

# ------------------------------------------------------------
# 3. Calculate basic information
# ------------------------------------------------------------

character_count = len(script)
word_count = len(script.split())

# ------------------------------------------------------------
# 4. Display information
# ------------------------------------------------------------

print("=" * 60)
print("LONG SCRIPT VALIDATION")
print("=" * 60)

print("\nCharacters:", character_count)
print("Words:", word_count)

print("\nScript preview:")
print("-" * 60)

preview = script[:500]

print(preview)

if len(script) > 500:
    print("\n...")

print("-" * 60)

print("\n✓ Script loaded successfully.")
print("✓ Script is ready for chunking.")

LONG SCRIPT VALIDATION

Characters: 423
Words: 67

Script preview:
------------------------------------------------------------
Most people think they understand what they see.

But perception is rarely as simple as it seems.
Our minds constantly fill in missing information,
make assumptions, and turn incomplete details into a story.

This is where things become interesting.
Because sometimes, what we believe we are seeing
has very little to do with what is actually there.

And the difference between perception and reality
can change everything.
------------------------------------------------------------

✓ Script loaded successfully.
✓ Script is ready for chunking.


# ============================================
# CELL 23 — SPLIT SCRIPT INTO NARRATION CHUNKS
# ============================================
## What this does:

This cell takes FULL_SCRIPT from Cell 22 and divides it into smaller pieces.

For now, we'll target roughly 80 words per chunk. Your current script is only 67 words, so it should produce one chunk.

Later, when you paste a much longer documentary script, this same cell will automatically divide it into manageable sections before we send them to Qwen3-TTS.

We split primarily at paragraph/sentence boundaries rather than cutting words randomly, so the narration should sound more natural.

## Expected Output:
For your current test script, you should see:

Total chunks: 1

followed by the complete script as Chunk 1

In [ ]:
import re

# ------------------------------------------------------------
# 1. Chunk settings
# ------------------------------------------------------------

MAX_WORDS_PER_CHUNK = 80

# ------------------------------------------------------------
# 2. Split script into paragraphs
# ------------------------------------------------------------

paragraphs = [
    paragraph.strip()
    for paragraph in re.split(r"\n\s*\n", script)
    if paragraph.strip()
]

# ------------------------------------------------------------
# 3. Split paragraphs into sentences
# ------------------------------------------------------------

sentences = []

for paragraph in paragraphs:
    paragraph_sentences = re.split(
        r"(?<=[.!?])\s+",
        paragraph
    )

    for sentence in paragraph_sentences:
        sentence = sentence.strip()

        if sentence:
            sentences.append(sentence)

# ------------------------------------------------------------
# 4. Build chunks
# ------------------------------------------------------------

chunks = []
current_chunk = []
current_word_count = 0

for sentence in sentences:
    sentence_word_count = len(sentence.split())

    # If adding this sentence exceeds the limit,
    # save the current chunk first.
    if (
        current_chunk
        and current_word_count + sentence_word_count > MAX_WORDS_PER_CHUNK
    ):
        chunks.append(" ".join(current_chunk))
        current_chunk = []
        current_word_count = 0

    current_chunk.append(sentence)
    current_word_count += sentence_word_count

# ------------------------------------------------------------
# 5. Save final chunk
# ------------------------------------------------------------

if current_chunk:
    chunks.append(" ".join(current_chunk))

# ------------------------------------------------------------
# 6. Display results
# ------------------------------------------------------------

print("=" * 60)
print("SCRIPT CHUNKING COMPLETE")
print("=" * 60)

print("\nTotal chunks:", len(chunks))
print("Maximum target words per chunk:", MAX_WORDS_PER_CHUNK)

for i, chunk in enumerate(chunks, start=1):
    print("\n" + "-" * 60)
    print(f"CHUNK {i}")
    print("-" * 60)

    print("Words:", len(chunk.split()))
    print(chunk)

print("\n" + "=" * 60)
print("✓ Script successfully divided into narration chunks.")
print("=" * 60)

SCRIPT CHUNKING COMPLETE

Total chunks: 1
Maximum target words per chunk: 80

------------------------------------------------------------
CHUNK 1
------------------------------------------------------------
Words: 67
Most people think they understand what they see. But perception is rarely as simple as it seems. Our minds constantly fill in missing information,
make assumptions, and turn incomplete details into a story. This is where things become interesting. Because sometimes, what we believe we are seeing
has very little to do with what is actually there. And the difference between perception and reality
can change everything.

✓ Script successfully divided into narration chunks.


# ============================================
# CELL 24 — GENERATE ALL NARRATION CHUNKS
# ============================================
##What this does:

- This cell takes the chunks created by Cell 23 and sends each chunk individually to Qwen3-TTS using your saved cold_analytical_male voice.

- Each chunk will be saved as its own WAV file inside the narration folder.

- This is the foundation for handling long documentaries: instead of asking the model to generate a 10-minute script in one huge request, we generate manageable pieces.

## Expected result:

- For your current 67-word test, there should be one generated WAV file:

```Qwen3TTS/
└── voices/
    └── cold_analytical_male/
        └── narrations/
            └── chunks/
                └── chunk_001.wav
```
- You should see a successful generation message for each chunk

In [ ]:
import shutil
import soundfile as sf
import torch

# ------------------------------------------------------------
# 1. Create chunk output folder
# ------------------------------------------------------------

CHUNKS_DIR = NARRATION_DIR / "chunks"
CHUNKS_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 2. Remove old test chunks
# ------------------------------------------------------------

for old_file in CHUNKS_DIR.glob("chunk_*.wav"):
    old_file.unlink()

# ------------------------------------------------------------
# 3. Generate each chunk
# ------------------------------------------------------------

print("=" * 60)
print("GENERATING NARRATION CHUNKS")
print("=" * 60)

generated_files = []

for i, chunk in enumerate(chunks, start=1):

    output_path = CHUNKS_DIR / f"chunk_{i:03d}.wav"

    print("\n" + "-" * 60)
    print(f"CHUNK {i}/{len(chunks)}")
    print("-" * 60)

    print("Words:", len(chunk.split()))
    print("Generating...")

    with torch.inference_mode():
        wavs, sr = clone_model.generate_voice_clone(
            text=chunk,
            language="English",
            voice_clone_prompt=gpu_prompt,
        )

    sf.write(
        output_path,
        wavs[0],
        sr
    )

    generated_files.append(output_path)

    duration = len(wavs[0]) / sr

    print("✓ Generated")
    print("  File:", output_path.name)
    print("  Duration:", round(duration, 2), "seconds")

# ------------------------------------------------------------
# 4. Summary
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("CHUNK GENERATION COMPLETE")
print("=" * 60)

print("\nGenerated files:")

for path in generated_files:
    print("  ✓", path)

print("\nTotal chunks generated:", len(generated_files))
print("✓ All narration chunks generated successfully.")

Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


GENERATING NARRATION CHUNKS

------------------------------------------------------------
CHUNK 1/1
------------------------------------------------------------
Words: 67
Generating...
✓ Generated
  File: chunk_001.wav
  Duration: 23.28 seconds

CHUNK GENERATION COMPLETE

Generated files:
  ✓ /content/drive/MyDrive/Qwen3TTS/voices/cold_analytical_male/narrations/chunks/chunk_001.wav

Total chunks generated: 1
✓ All narration chunks generated successfully.


# ============================================
# CELL 25 — COMBINE NARRATION CHUNKS
# ============================================
## What this does:

- This cell takes every chunk_XXX.wav generated by Cell 24, puts them in numerical order, and combines them into one complete narration WAV.

- For your current test, there is only one chunk, so the final narration will contain that single chunk.

- Later, if your documentary produces 20 or 30 chunks, this cell will combine all of them automatically.

## Expected result:

- You should get:

```Qwen3TTS/
└── voices/
    └── cold_analytical_male/
        └── narrations/
            ├── chunks/
            │   └── chunk_001.wav
            └── final_narration.wav
```
- The output should show the number of chunks combined and the final duration.

In [ ]:
import numpy as np
import soundfile as sf

# ------------------------------------------------------------
# 1. Find generated chunks
# ------------------------------------------------------------

chunk_files = sorted(
    CHUNKS_DIR.glob("chunk_*.wav")
)

if not chunk_files:
    raise FileNotFoundError(
        "No narration chunks found. Run Cell 24 first."
    )

# ------------------------------------------------------------
# 2. Load all chunks
# ------------------------------------------------------------

print("=" * 60)
print("COMBINING NARRATION CHUNKS")
print("=" * 60)

audio_parts = []
sample_rate = None

for i, chunk_path in enumerate(chunk_files, start=1):

    print(f"\nLoading chunk {i}/{len(chunk_files)}:")
    print(chunk_path.name)

    audio, sr = sf.read(chunk_path)

    # Make sure every chunk uses the same sample rate.
    if sample_rate is None:
        sample_rate = sr
    elif sr != sample_rate:
        raise ValueError(
            f"Sample rate mismatch: "
            f"{chunk_path.name} uses {sr} Hz, "
            f"expected {sample_rate} Hz."
        )

    audio_parts.append(audio)

    print("  ✓ Loaded")
    print("  Duration:", round(len(audio) / sr, 2), "seconds")

# ------------------------------------------------------------
# 3. Combine chunks
# ------------------------------------------------------------

combined_audio = np.concatenate(audio_parts)

# ------------------------------------------------------------
# 4. Save final narration
# ------------------------------------------------------------

FINAL_NARRATION_PATH = NARRATION_DIR / "final_narration.wav"

sf.write(
    FINAL_NARRATION_PATH,
    combined_audio,
    sample_rate
)

# ------------------------------------------------------------
# 5. Display summary
# ------------------------------------------------------------

final_duration = len(combined_audio) / sample_rate

print("\n" + "=" * 60)
print("NARRATION COMBINATION COMPLETE")
print("=" * 60)

print("\nChunks combined:", len(chunk_files))
print("Sample rate:", sample_rate)
print("Final duration:", round(final_duration, 2), "seconds")

print("\nSaved to:")
print(FINAL_NARRATION_PATH)

print("\n✓ Final narration created successfully.")

COMBINING NARRATION CHUNKS

Loading chunk 1/1:
chunk_001.wav
  ✓ Loaded
  Duration: 23.28 seconds

NARRATION COMBINATION COMPLETE

Chunks combined: 1
Sample rate: 24000
Final duration: 23.28 seconds

Saved to:
/content/drive/MyDrive/Qwen3TTS/voices/cold_analytical_male/narrations/final_narration.wav

✓ Final narration created successfully.


```We now have the complete narration pipeline:

FULL_SCRIPT
    ↓
Cell 23 — Split into chunks
    ↓
Cell 24 — Generate each chunk
    ↓
Cell 25 — Combine chunks
    ↓
final_narration.wav
```

- Before we start making longer scripts, let's add one useful cell to listen to the actual final combined file. This also confirms we're listening to exactly what will eventually be used in the video.

# ============================================
# CELL 26 — PLAY FINAL NARRATION
# ============================================
## What this does:

- This cell plays final_narration.wav directly in Colab.

- It does not generate anything, use the GPU, or modify the audio. It simply lets us hear the final combined narration produced by Cells 23–25.

## Expected result:

- An audio player should appear below the cell. Press Play and listen to the complete narration.

In [ ]:
from IPython.display import Audio, display

print("=" * 60)
print("PLAYING FINAL NARRATION")
print("=" * 60)

display(
    Audio(
        str(FINAL_NARRATION_PATH),
        autoplay=False
    )
)

PLAYING FINAL NARRATION


# ============================================
# CELL 27 — CREATE REUSABLE NARRATION FUNCTION
# ============================================
## What this does:

- This cell turns the work we've done into a single reusable function.

- Instead of manually running separate generation code every time, we'll eventually be able to give the function a script and it will:
```
Split the script into chunks.
Generate each chunk using cold_analytical_male.
Save each chunk.
Combine all chunks.
Save one final WAV file.
```

- We're not optimizing pauses, emotions, or delivery yet. This is our stable baseline.

## Expected result:

- The cell itself should only define the function. It should not generate any audio yet.

- You should see:
```
✓ Narration function ready.
```

In [ ]:
import re
import numpy as np
import soundfile as sf
import torch


def generate_narration(
    text,
    output_name="narration.wav",
    max_words_per_chunk=80,
):
    """
    Generate a complete narration using the saved voice clone.

    The script is split into manageable chunks, each chunk
    is generated separately, and the chunks are combined
    into one final WAV file.
    """

    # --------------------------------------------------------
    # 1. Validate input
    # --------------------------------------------------------

    text = text.strip()

    if not text:
        raise ValueError("Narration text cannot be empty.")

    # --------------------------------------------------------
    # 2. Split into paragraphs
    # --------------------------------------------------------

    paragraphs = [
        paragraph.strip()
        for paragraph in re.split(r"\n\s*\n", text)
        if paragraph.strip()
    ]

    # --------------------------------------------------------
    # 3. Split paragraphs into sentences
    # --------------------------------------------------------

    sentences = []

    for paragraph in paragraphs:

        paragraph_sentences = re.split(
            r"(?<=[.!?])\s+",
            paragraph
        )

        for sentence in paragraph_sentences:

            sentence = sentence.strip()

            if sentence:
                sentences.append(sentence)

    # --------------------------------------------------------
    # 4. Build chunks
    # --------------------------------------------------------

    narration_chunks = []

    current_chunk = []
    current_word_count = 0

    for sentence in sentences:

        sentence_word_count = len(sentence.split())

        if (
            current_chunk
            and current_word_count + sentence_word_count
            > max_words_per_chunk
        ):
            narration_chunks.append(
                " ".join(current_chunk)
            )

            current_chunk = []
            current_word_count = 0

        current_chunk.append(sentence)
        current_word_count += sentence_word_count

    if current_chunk:
        narration_chunks.append(
            " ".join(current_chunk)
        )

    # --------------------------------------------------------
    # 5. Create output folders
    # --------------------------------------------------------

    output_dir = NARRATION_DIR / Path(output_name).stem
    chunks_dir = output_dir / "chunks"

    chunks_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    # --------------------------------------------------------
    # 6. Generate chunks
    # --------------------------------------------------------

    print("=" * 60)
    print("GENERATING NARRATION")
    print("=" * 60)

    print("\nTotal chunks:", len(narration_chunks))

    generated_audio = []
    sample_rate = None

    for i, chunk in enumerate(
        narration_chunks,
        start=1
    ):

        print("\n" + "-" * 60)
        print(
            f"CHUNK {i}/{len(narration_chunks)}"
        )
        print("-" * 60)

        print("Words:", len(chunk.split()))
        print("Generating...")

        with torch.inference_mode():

            wavs, sr = clone_model.generate_voice_clone(
                text=chunk,
                language="English",
                voice_clone_prompt=gpu_prompt,
            )

        if sample_rate is None:
            sample_rate = sr

        elif sr != sample_rate:
            raise ValueError(
                "Sample rate mismatch between chunks."
            )

        chunk_path = (
            chunks_dir
            / f"chunk_{i:03d}.wav"
        )

        sf.write(
            chunk_path,
            wavs[0],
            sr
        )

        generated_audio.append(wavs[0])

        duration = len(wavs[0]) / sr

        print("✓ Generated")
        print("File:", chunk_path.name)
        print(
            "Duration:",
            round(duration, 2),
            "seconds"
        )

    # --------------------------------------------------------
    # 7. Combine audio
    # --------------------------------------------------------

    final_audio = np.concatenate(
        generated_audio
    )

    final_path = (
        output_dir
        / output_name
    )

    sf.write(
        final_path,
        final_audio,
        sample_rate
    )

    # --------------------------------------------------------
    # 8. Final information
    # --------------------------------------------------------

    final_duration = (
        len(final_audio)
        / sample_rate
    )

    print("\n" + "=" * 60)
    print("NARRATION COMPLETE")
    print("=" * 60)

    print("\nSaved to:")
    print(final_path)

    print(
        "\nDuration:",
        round(final_duration, 2),
        "seconds"
    )

    print(
        "Chunks:",
        len(narration_chunks)
    )

    print(
        "\n✓ Narration generated successfully."
    )

    return final_path

# ============================================
# CELL 28 — TEST REUSABLE NARRATION FUNCTION
# ============================================
## What this does:

- This cell actually calls the function we created in Cell 27.

 - We'll use a fresh short script to make sure the entire reusable workflow works from start to finish:
```
script → chunking → voice generation → WAV → final file
```
- This is still only a test. We're not using a real documentary script yet.

## Expected result:

- You should see something similar to:

```
Total chunks: 1

CHUNK 1/1
Words: ...
Generating...
✓ Generated

============================================================
NARRATION COMPLETE
============================================================

Saved to:
.../narrations/function_test/narration_test.wav

Duration: ... seconds
Chunks: 1

✓ Narration generated successfully.
```

In [ ]:
TEST_NARRATION = """
There are moments when the obvious answer is not the correct one.
Sometimes, the truth is hidden in the details we choose to ignore.
And once you notice those details, everything begins to look different.
"""

TEST_OUTPUT = generate_narration(
    text=TEST_NARRATION,
    output_name="narration_test.wav",
    max_words_per_chunk=80,
)

print("\nFinal file:")
print(TEST_OUTPUT)

Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


GENERATING NARRATION

Total chunks: 1

------------------------------------------------------------
CHUNK 1/1
------------------------------------------------------------
Words: 35
Generating...
✓ Generated
File: chunk_001.wav
Duration: 11.52 seconds

NARRATION COMPLETE

Saved to:
/content/drive/MyDrive/Qwen3TTS/voices/cold_analytical_male/narrations/narration_test/narration_test.wav

Duration: 11.52 seconds
Chunks: 1

✓ Narration generated successfully.

Final file:
/content/drive/MyDrive/Qwen3TTS/voices/cold_analytical_male/narrations/narration_test/narration_test.wav


```
Script
  ↓
Automatic chunking
  ↓
Voice cloning
  ↓
Individual WAV chunk
  ↓
Final combined WAV
```

# ============================================
# CELL 29 — PLAY REUSABLE NARRATION
# ============================================
## What this does:

- This cell gives us a simple way to listen to the output of generate_narration().

- Instead of manually finding WAV files in Google Drive, we can pass the path returned by the function and listen to it directly in Colab.

- It does not generate or modify anything.

## Expected result:

- An audio player should appear below the cell containing the narration_test.wav generated by Cell 28.

In [ ]:
from IPython.display import Audio, display

print("=" * 60)
print("PLAYING REUSABLE NARRATION")
print("=" * 60)

display(
    Audio(
        str(TEST_OUTPUT),
        autoplay=False
    )
)

PLAYING REUSABLE NARRATION


# ============================================
# CELL 30 — NATURAL PAUSE & PUNCTUATION TEST
# ============================================
## What this does:

- This cell tests how the Qwen voice responds to different punctuation and sentence structures.

- We'll deliberately use:
```
, for small pauses
. for normal pauses
... for longer/dramatic pauses
! for emphasis
? for questioning tone
Separate lines for stronger separation
```
- We're not changing the reusable narration function yet. This is only an experiment so we can hear what Qwen naturally does with punctuation.

## Expected result:

- One WAV file should be generated:
```
Qwen3TTS/
└── voices/
    └── cold_analytical_male/
        └── narrations/
            └── punctuation_test/
                └── punctuation_test.wav
```
- The cell will also print the duration.

In [ ]:
PUNCTUATION_TEST = """
Most people think they understand what they see.

But do they?

Usually, they only understand what they expect to see...

And that is where things become interesting.

Because perception can be misleading, sometimes very misleading.

The truth is simple!

We do not always see reality.

We see what our minds expect to see.
"""

print("=" * 60)
print("NATURAL PAUSE & PUNCTUATION TEST")
print("=" * 60)

print("\nTest text:")
print(PUNCTUATION_TEST)

# ------------------------------------------------------------
# Generate test audio
# ------------------------------------------------------------

with torch.inference_mode():

    wavs, sr = clone_model.generate_voice_clone(
        text=PUNCTUATION_TEST.strip(),
        language="English",
        voice_clone_prompt=gpu_prompt,
    )

# ------------------------------------------------------------
# Save test audio
# ------------------------------------------------------------

PUNCTUATION_TEST_DIR = (
    NARRATION_DIR / "punctuation_test"
)

PUNCTUATION_TEST_DIR.mkdir(
    parents=True,
    exist_ok=True
)

PUNCTUATION_TEST_PATH = (
    PUNCTUATION_TEST_DIR
    / "punctuation_test.wav"
)

sf.write(
    PUNCTUATION_TEST_PATH,
    wavs[0],
    sr
)

# ------------------------------------------------------------
# Display result
# ------------------------------------------------------------

duration = len(wavs[0]) / sr

print("\n✓ Generation complete.")

print("\nSaved to:")
print(PUNCTUATION_TEST_PATH)

print("\nSample rate:", sr)
print("Duration:", round(duration, 2), "seconds")

Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


NATURAL PAUSE & PUNCTUATION TEST

Test text:

Most people think they understand what they see.

But do they?

Usually, they only understand what they expect to see...

And that is where things become interesting.

Because perception can be misleading, sometimes very misleading.

The truth is simple!

We do not always see reality.

We see what our minds expect to see.


✓ Generation complete.

Saved to:
/content/drive/MyDrive/Qwen3TTS/voices/cold_analytical_male/narrations/punctuation_test/punctuation_test.wav

Sample rate: 24000
Duration: 21.44 seconds


# ============================================
# CELL 31 — PLAY PUNCTUATION TEST
# ============================================

# What this does:

- This cell plays the punctuation_test.wav that Cell 30 already generated.

- It does not generate anything again, so it won't use the GPU or take another inference run.

## Expected result:

- An audio player should appear below the cell. Listen specifically to:
```
"But do they?" → does the voice sound questioning?
"expect to see..." → does it create a longer pause?
"The truth is simple!" → does it add emphasis?
The blank lines → do they create natural separation?
```

In [ ]:
from IPython.display import Audio, display

print("=" * 60)
print("PLAYING PUNCTUATION TEST")
print("=" * 60)

display(
    Audio(
        str(PUNCTUATION_TEST_PATH),
        autoplay=False
    )
)

PLAYING PUNCTUATION TEST


# ============================================
# CELL 32 — PUNCTUATION COMPARISON TEST
# ============================================
## What this does:

- This cell generates four very short versions of essentially the same sentence, each using different punctuation.

- We'll compare:
```
A normal statement .
A question ?
An ellipsis ...
An exclamation !
```
- This makes it much easier to hear exactly how Qwen changes its delivery instead of judging several punctuation types inside one paragraph.

## Expected result:

- Four WAV files will be created:
```
punctuation_test/
├── normal.wav
├── question.wav
├── ellipsis.wav
└── exclamation.wav
```
- No existing files will be overwritten.

In [ ]:
import soundfile as sf
import torch

# ------------------------------------------------------------
# 1. Test sentences
# ------------------------------------------------------------

PUNCTUATION_VARIANTS = {
    "normal": "The truth is hidden in the details.",
    "question": "The truth is hidden in the details?",
    "ellipsis": "The truth is hidden in the details...",
    "exclamation": "The truth is hidden in the details!",
}

# ------------------------------------------------------------
# 2. Create output folder
# ------------------------------------------------------------

PUNCTUATION_COMPARE_DIR = (
    NARRATION_DIR / "punctuation_comparison"
)

PUNCTUATION_COMPARE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------------------------
# 3. Generate each version
# ------------------------------------------------------------

print("=" * 60)
print("PUNCTUATION COMPARISON TEST")
print("=" * 60)

generated_paths = {}

for name, text in PUNCTUATION_VARIANTS.items():

    print("\n" + "-" * 60)
    print(name.upper())
    print("-" * 60)

    print("Text:", text)
    print("Generating...")

    with torch.inference_mode():

        wavs, sr = clone_model.generate_voice_clone(
            text=text,
            language="English",
            voice_clone_prompt=gpu_prompt,
        )

    output_path = (
        PUNCTUATION_COMPARE_DIR
        / f"{name}.wav"
    )

    sf.write(
        output_path,
        wavs[0],
        sr
    )

    generated_paths[name] = output_path

    duration = len(wavs[0]) / sr

    print("✓ Generated")
    print("Duration:", round(duration, 2), "seconds")

print("\n" + "=" * 60)
print("✓ ALL FOUR TESTS GENERATED")
print("=" * 60)

Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


PUNCTUATION COMPARISON TEST

------------------------------------------------------------
NORMAL
------------------------------------------------------------
Text: The truth is hidden in the details.
Generating...


Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


✓ Generated
Duration: 2.56 seconds

------------------------------------------------------------
QUESTION
------------------------------------------------------------
Text: The truth is hidden in the details?
Generating...


Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


✓ Generated
Duration: 2.32 seconds

------------------------------------------------------------
ELLIPSIS
------------------------------------------------------------
Text: The truth is hidden in the details...
Generating...


Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


✓ Generated
Duration: 2.4 seconds

------------------------------------------------------------
EXCLAMATION
------------------------------------------------------------
Text: The truth is hidden in the details!
Generating...
✓ Generated
Duration: 2.4 seconds

✓ ALL FOUR TESTS GENERATED


# ============================================
# CELL 33 — PLAY PUNCTUATION COMPARISON
# ============================================
## What this does:

- This cell creates four separate audio players, one for each version from Cell 32.

- You'll be able to play them one after another and directly compare how Qwen handles ., ?, ..., and !.

## Expected result:

- You should see four audio players labeled:
```
NORMAL
QUESTION
ELLIPSIS
EXCLAMATION
```
- Play them in that order and listen specifically to the ending of the sentence.

In [ ]:
from IPython.display import Audio, display

print("=" * 60)
print("PUNCTUATION COMPARISON")
print("=" * 60)

for name, path in generated_paths.items():

    print("\n" + "-" * 60)
    print(name.upper())
    print("-" * 60)

    display(
        Audio(
            str(path),
            autoplay=False
        )
    )

PUNCTUATION COMPARISON

------------------------------------------------------------
NORMAL
------------------------------------------------------------



------------------------------------------------------------
QUESTION
------------------------------------------------------------



------------------------------------------------------------
ELLIPSIS
------------------------------------------------------------



------------------------------------------------------------
EXCLAMATION
------------------------------------------------------------


# ============================================
# CELL 34 — NATURAL INTONATION TEST
# ============================================
## What this does:

- This cell tests punctuation using sentences where the punctuation actually makes sense naturally.

- We'll test:
```
Statement: "The truth is hidden in the details."
Question: "Does the truth hide in the details?"
Ellipsis: "The truth is hidden in the details... or is it?"
Exclamation: "The truth is hidden in the details!"
```
- The goal isn't to make the voice shout. We're checking whether Qwen naturally changes intonation, emphasis, rhythm, and pauses according to the meaning.

## Expected result:

- Four new WAV files will be generated:
```
punctuation_comparison/
├── statement.wav
├── question.wav
├── ellipsis.wav
└── exclamation.wav
```
- Then Cell 35 will give us clearly labeled players containing the exact sentences.

In [ ]:
import soundfile as sf
import torch

# ------------------------------------------------------------
# 1. Natural sentences
# ------------------------------------------------------------

NATURAL_INTONATION_VARIANTS = {
    "statement": (
        "The truth is hidden in the details."
    ),

    "question": (
        "Does the truth hide in the details?"
    ),

    "ellipsis": (
        "The truth is hidden in the details... or is it?"
    ),

    "exclamation": (
        "The truth is hidden in the details!"
    ),
}

# ------------------------------------------------------------
# 2. Create output folder
# ------------------------------------------------------------

NATURAL_INTONATION_DIR = (
    NARRATION_DIR / "natural_intonation_test"
)

NATURAL_INTONATION_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------------------------
# 3. Generate each version
# ------------------------------------------------------------

print("=" * 60)
print("NATURAL INTONATION TEST")
print("=" * 60)

natural_test_paths = {}

for name, text in NATURAL_INTONATION_VARIANTS.items():

    print("\n" + "-" * 60)
    print(name.upper())
    print("-" * 60)

    print("Text:", text)
    print("Generating...")

    with torch.inference_mode():

        wavs, sr = clone_model.generate_voice_clone(
            text=text,
            language="English",
            voice_clone_prompt=gpu_prompt,
        )

    output_path = (
        NATURAL_INTONATION_DIR
        / f"{name}.wav"
    )

    sf.write(
        output_path,
        wavs[0],
        sr
    )

    natural_test_paths[name] = output_path

    duration = len(wavs[0]) / sr

    print("✓ Generated")
    print("Duration:", round(duration, 2), "seconds")

print("\n" + "=" * 60)
print("✓ ALL NATURAL INTONATION TESTS GENERATED")
print("=" * 60)

Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


NATURAL INTONATION TEST

------------------------------------------------------------
STATEMENT
------------------------------------------------------------
Text: The truth is hidden in the details.
Generating...


Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


✓ Generated
Duration: 2.4 seconds

------------------------------------------------------------
QUESTION
------------------------------------------------------------
Text: Does the truth hide in the details?
Generating...


Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


✓ Generated
Duration: 2.64 seconds

------------------------------------------------------------
ELLIPSIS
------------------------------------------------------------
Text: The truth is hidden in the details... or is it?
Generating...


Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


✓ Generated
Duration: 3.44 seconds

------------------------------------------------------------
EXCLAMATION
------------------------------------------------------------
Text: The truth is hidden in the details!
Generating...
✓ Generated
Duration: 2.64 seconds

✓ ALL NATURAL INTONATION TESTS GENERATED


Reviewing the results


In [ ]:
from IPython.display import Audio, display, Markdown

print("=" * 60)
print("NATURAL INTONATION TEST — AUDIO PLAYERS")
print("=" * 60)

# ------------------------------------------------------------
# 1. Display each sentence directly above its audio player
# ------------------------------------------------------------

for name, text in NATURAL_INTONATION_VARIANTS.items():

    print("\n" + "-" * 60)
    print(name.upper())
    print("-" * 60)

    # Exact sentence shown above the player
    display(Markdown(f"### {text}"))

    # Audio player
    output_path = natural_test_paths[name]

    display(
        Audio(
            filename=str(output_path),
            autoplay=False
        )
    )

print("\n" + "=" * 60)
print("✓ ALL NATURAL INTONATION PLAYERS READY")
print("=" * 60)

NATURAL INTONATION TEST — AUDIO PLAYERS

------------------------------------------------------------
STATEMENT
------------------------------------------------------------


### The truth is hidden in the details.


------------------------------------------------------------
QUESTION
------------------------------------------------------------


### Does the truth hide in the details?


------------------------------------------------------------
ELLIPSIS
------------------------------------------------------------


### The truth is hidden in the details... or is it?


------------------------------------------------------------
EXCLAMATION
------------------------------------------------------------


### The truth is hidden in the details!


✓ ALL NATURAL INTONATION PLAYERS READY
